# Beam-scan raster: HFSS beam against a uniform sky and the canyon terrain

**Question.** As the antenna points through the 07-17 elevation-sweep raster, how much of the HFSS beam lands on terrain, and what antenna temperature does a uniform 1000 K sky above a 300 K canyon give at each time and frequency?

**Inputs.**
- **Data:** box-air (input 4), `RFANT` rows, `curation/pointing_table.parquet` quality `ok`, 07-17 20:26:00–21:28:40 UTC. Reversal glitches in the solved elevation are dropped, then the rows are decimated ×8 in time.
- **RFI flags:** `flags/v3-beta.1/flags_20260717.h5`, input 4, per file and row.
- **Pointing:** the table's `az_deg` (commanded azimuth) and `el_deg` (IMU elevation).
- **Terrain:** `curation/horizon_profiles_v0002` (DEM ray trace from the v0001 antenna position), 91 m era.
- **Beam:** HFSS `bowtie_beam.npz` via `eigsep_data.beam_sim.read_beam`: 51 slices, 50.8–246.1 MHz, total power $|E|^2$.

**Pointing convention (Aaron, 2026-09-25).** The HFSS body frame has its boresight along +z and the dipole arm along +x; the notebook checks this in §2. Body to topocentric ENU is

$$\hat v_\mathrm{ENU} = R_z(\psi)\,R_x(\mathrm{el})\,R_z(\mathrm{az})\,\hat v_\mathrm{body},\qquad \psi = 142.164^\circ.$$

- **az** rolls the antenna about its boresight, right-handed. At az = 0 the arm lies along the highline, and $\psi$ is the highline's direction counter-clockwise from East (GPS anchors, ±0.6°).
- **el** tips the boresight about the highline axis: el = 0 is zenith, +90° the horizon toward compass 37.836°, +180° nadir.

This is the convention of `known_quantities` v0001 (arm along the highline at az = 0; boresight sweep plane at compass 37.836°). The `fit_beam.py` beam fits use a different one, which Aaron has said is wrong; fixing it is separate work.

**Model.** $T_\mathrm{ant}(t,\nu) = \int P_\nu\,[1000\,\mathrm{K}\cdot(1-M_t) + 300\,\mathrm{K}\cdot M_t]\,d\Omega \,/ \int P_\nu\,d\Omega$. Here $M_t$ is 1 where a direction lies below the horizon profile at its bearing. The integral runs at the HFSS frequencies and is then interpolated linearly onto the correlator channels. There is no ground plane, receiver or gain: this is an antenna temperature, not a prediction of correlator counts.

In [ ]:
from pathlib import Path
import json
import os

import h5py
import healpy as hp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
import pyarrow.parquet as pq

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)
plt.rcParams.update({'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'marjum-2026-07/data').is_dir())
CAMPAIGN = ROOT / 'marjum-2026-07'
import eigsep_data
from eigsep_data import AntennaResolutionPolicy, MetadataIndex
from eigsep_data.beam_sim import read_beam
eigsep_data.set_campaign_root(CAMPAIGN)

RASTER_UTC = ('2026-07-17 20:26:00', '2026-07-17 21:28:40')
DECIMATE = 8
T_SKY, T_TERRAIN = 1000.0, 300.0            # K
PSI_DEG = 142.164                           # highline direction, deg ccw from East (known_quantities v0001)
ERA = '91m'
NSIDE_INT = 64                              # integration grid (HFSS native is 32)
DF = 250.0 / 1024
FREQ = np.arange(1024) * DF
HORIZON = CAMPAIGN / 'curation/horizon_profiles_v0002'
FLAGS = CAMPAIGN / 'flags/v3-beta.1/flags_20260717.h5'

## 1. Data, pointing and RFI flags

In [ ]:
t0, t1 = (pd.Timestamp(s, tz='UTC').timestamp() for s in RASTER_UTC)
policy = AntennaResolutionPolicy.load(CAMPAIGN / 'curation/antenna_resolution.json')
bundle = MetadataIndex(CAMPAIGN / 'data').select(time=(t0, t1)).load_bundle(
    antenna='box-air', missing='skip', resolution_policy=policy)
assert np.allclose(bundle.freqs_mhz, FREQ)
meta = bundle.meta.reset_index(drop=True).assign(t=lambda x: x.time_best.astype(float))
point = pq.read_table(CAMPAIGN / 'curation/pointing_table.parquet',
                      columns=['t_utc_s', 'az_deg', 'el_deg', 'quality', 'height_era'],
                      filters=[('t_utc_s', '>=', t0 - 60), ('t_utc_s', '<=', t1 + 60)]).to_pandas()
POINT_PROV = json.loads(pq.read_schema(CAMPAIGN / 'curation/pointing_table.parquet').metadata[b'eigsep_provenance'])
j = pd.merge_asof(meta.reset_index(names='bundle_row').sort_values('t'), point.sort_values('t_utc_s'),
                  left_on='t', right_on='t_utc_s', direction='nearest', tolerance=0.35)
j = j[(j.quality == 'ok') & (j.rfswitch == 'RFANT')].reset_index(drop=True)
# The raster is entirely in the 91 m era; a few rows carry a blank era label in the table.
assert set(j.height_era) <= {'~91m', ''}, set(j.height_era)
print('rows with a blank height_era: %d of %d' % ((j.height_era == '').sum(), len(j)))


def reversal_glitches(el_deg, tol=15.0):
    """Isolated solved-elevation faults at sweep reversals (see beam_explorer / fit_beam)."""
    el = np.asarray(el_deg, float)
    close = np.zeros(el.size, int)
    for k in (-2, -1, 1, 2):
        d = np.abs((el - np.roll(el, k) + 180.0) % 360.0 - 180.0)
        d[slice(0, -k) if k < 0 else slice(-k, None)] = 0.0
        close += d <= tol
    return close < 3


glitch = reversal_glitches(j.el_deg.to_numpy())
j = j[~glitch].reset_index(drop=True).iloc[::DECIMATE].reset_index(drop=True)
D = np.asarray(bundle.data[j.bundle_row.to_numpy()], float)          # counts, (time, 1024)
AZ, EL = j.az_deg.to_numpy(), j.el_deg.to_numpy()
TIME = pd.DatetimeIndex(pd.to_datetime(j.t.to_numpy(), unit='s', utc=True))
print('pointing: %s' % POINT_PROV['compact'])
print('%d rows after dropping %d glitches and decimating x%d; %d files' % (len(j), glitch.sum(), DECIMATE, j.file.nunique()))

# RFI flags: bitfield per (file, row, channel) over the flag product's channels.
bits = {b['name']: b for b in json.loads((FLAGS.parent / 'flag_bits.json').read_text())['bits']}
RFI_BITS = sum(b['value'] for b in bits.values() if b['rfi'])
FLAG = np.zeros(D.shape, np.uint16)
HAS_FLAGS = np.zeros(len(j), bool)
missing = []
with h5py.File(FLAGS) as h:
    fch = np.round(h['freqs_mhz'][:] / DF).astype(int)
    for f, g in j.groupby('file'):
        if 'mask/%s/4' % f not in h:
            missing.append(f)
            continue
        FLAG[np.ix_(g.index.to_numpy(), fch)] = h['mask/%s/4' % f][:][g.row.to_numpy()]
        HAS_FLAGS[g.index.to_numpy()] = True
    outside = np.setdiff1d(np.arange(1024), fch)
# The v3-beta.1 run failed on the batches covering 20:54-21:28 (DPSS conjugate-gradient solve did not
# converge; it had rejected all but 11% and 3% of the data; flags/v3-beta.1/run_reports). Those rows
# have no flags: they are kept, unmasked, and marked by HAS_FLAGS.
print('%d of %d files have no v3-beta.1 flags (%s ... %s): %d rows unflagged'
      % (len(missing), j.file.nunique(), missing[0] if missing else '-', missing[-1] if missing else '-', (~HAS_FLAGS).sum()))
FLAG[:, outside] = bits['invalid_input_or_domain']['value']
RFI = (FLAG & RFI_BITS) != 0
ANY = FLAG != 0
band = (FREQ >= 50) & (FREQ <= 246)
fl = HAS_FLAGS[:, None] & band[None, :]
print('flags over 50-246 MHz, rows with flags: any bit %.1f%%, RFI bits %.1f%%' % (100 * ANY[fl].mean(), 100 * RFI[fl].mean()))
display(pd.Series({n: 100 * ((FLAG[fl] & b['value']) != 0).mean() for n, b in bits.items()},
                  name='% of flagged-row samples, 50-246 MHz').round(2).to_frame())

## 2. Beam, terrain and the rotation

The HFSS body frame is checked before use: the power should peak along ±z (front/back symmetric), the E-field at +z should lie along x, and there should be a null along ±x (the dipole axis). The power pattern is upgraded from nside 32 to 64, so that the 0.1° horizon profile is sampled more finely than the HFSS pixels. The beam itself is smooth at that scale.

In [ ]:
BEAM_CART, _, HFSS_FREQS = read_beam()
NSIDE = hp.npix2nside(BEAM_CART.shape[-1])
POWER32 = np.sum(np.abs(BEAM_CART) ** 2, axis=1)                    # (51, npix), |E|^2
k150 = int(np.argmin(abs(HFSS_FREQS - 150)))
power_at = lambda v: POWER32[k150, hp.vec2pix(NSIDE, *v)] / POWER32[k150].max()
e_zen = np.abs(BEAM_CART[k150, :, hp.vec2pix(NSIDE, 0, 0, 1)])
print('150 MHz power (rel. to peak): +z %.3f  -z %.3f  +x %.4f  +y %.3f;  |E| at +z (x, y, z) = %s'
      % (power_at((0, 0, 1)), power_at((0, 0, -1)), power_at((1, 0, 0)), power_at((0, 1, 0)), np.round(e_zen / e_zen.max(), 4)))
assert power_at((0, 0, 1)) > 0.9 and power_at((0, 0, -1)) > 0.9 and power_at((1, 0, 0)) < 0.05
assert e_zen[0] > 10 * max(e_zen[1], e_zen[2])

POWER = np.array([hp.ud_grade(p, NSIDE_INT, power=-2) for p in POWER32])  # preserves the integral
POWER /= POWER.sum(axis=1, keepdims=True)
VBODY = np.array(hp.pix2vec(NSIDE_INT, np.arange(hp.nside2npix(NSIDE_INT))))  # (3, npix)

with np.load(HORIZON.with_suffix('.npz')) as z:
    HZ_BEAR = z['bearings_deg']
    HZ_ELEV = z['elev_rad_%s' % ERA]
HZ_META = json.loads(HORIZON.with_suffix('.json').read_text())
print('horizon: %s, %s era; elevation %.2f-%.2f deg' % (HZ_META['stamp'], ERA, *np.degrees([HZ_ELEV.min(), HZ_ELEV.max()])))


from eigsep_base.rotations import mount_rotation


def body_to_enu(az_deg, el_deg):
    """Highline roll mount, ENU = Rz(psi) Rx(el) Rz(az) body: the convention of the beam fits."""
    return mount_rotation(az_deg, el_deg, PSI_DEG)


# Convention checks: el=0 boresight is zenith; el=90 boresight points to compass 37.836 deg
# (ENU bearing 52.164 deg ccw from East); at az=0 the arm lies along the highline.
b90 = body_to_enu(0, 90) @ [0, 0, 1]
arm0 = body_to_enu(0, 0) @ [1, 0, 0]
print('el=90 boresight: ENU %s, compass bearing %.3f deg' % (np.round(b90, 4), np.degrees(np.arctan2(b90[0], b90[1])) % 360))
print('az=0 arm: ENU %s, direction %.3f deg ccw from East' % (np.round(arm0, 4), np.degrees(np.arctan2(arm0[1], arm0[0])) % 360))
assert np.allclose(body_to_enu(0, 0) @ [0, 0, 1], [0, 0, 1])
assert abs(np.degrees(np.arctan2(b90[0], b90[1])) % 360 - 37.836) < 1e-6


def terrain_mask(az_deg, el_deg):
    v = body_to_enu(az_deg, el_deg) @ VBODY                          # ENU unit vectors of body pixels
    elev = np.arcsin(np.clip(v[2], -1, 1))
    bearing = np.degrees(np.arctan2(v[1], v[0])) % 360.0               # ccw from East, as the profile
    horizon = np.interp(bearing, HZ_BEAR, HZ_ELEV, period=360.0)
    return elev < horizon

## 3. Terrain fraction and antenna temperature

In [ ]:
F_TERRAIN = np.array([POWER @ terrain_mask(a, e) for a, e in zip(AZ, EL)])     # (time, 51)
T_HFSS = T_SKY * (1 - F_TERRAIN) + T_TERRAIN * F_TERRAIN                     # K, at HFSS frequencies
T_ANT = np.array([np.interp(FREQ, HFSS_FREQS, row, left=np.nan, right=np.nan) for row in T_HFSS])  # (time, 1024)
print('terrain fraction: min %.3f  median %.3f  max %.3f' % (F_TERRAIN.min(), np.median(F_TERRAIN), F_TERRAIN.max()))
print('T_ant [K]: min %.1f  median %.1f  max %.1f' % (T_HFSS.min(), np.median(T_HFSS), T_HFSS.max()))

# Sanity: pointing at zenith with the front/back-symmetric beam, about half the power is below the horizon.
print('zenith pointing, 150 MHz terrain fraction: %.3f' % (POWER[k150] @ terrain_mask(0.0, 0.0)))

In [ ]:
tnum = mdates.date2num(TIME.tz_localize(None))
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, constrained_layout=True)
axes[0].plot(TIME, (AZ + 180) % 360 - 180, '.', ms=2, label='az (commanded)')
axes[0].plot(TIME, EL, '.', ms=2, label='el (IMU)')
axes[0].set_ylabel('deg'); axes[0].legend(markerscale=5)
for mhz in (60, 100, 150, 200, 240):
    k = int(np.argmin(abs(HFSS_FREQS - mhz)))
    axes[1].plot(TIME, F_TERRAIN[:, k], lw=0.8, label='%.1f MHz' % HFSS_FREQS[k])
    axes[2].plot(TIME, T_HFSS[:, k], lw=0.8)
axes[1].set_ylabel('terrain fraction'); axes[1].legend(fontsize=9, ncol=5)
axes[2].set_ylabel(r'$T_\mathrm{ant}$ [K]'); axes[2].set_xlabel('Time [UTC]')
axes[0].set_title('Raster pointing, beam-weighted terrain fraction, and antenna temperature')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
for ax, mhz in zip(axes, (100, 200)):
    k = int(np.argmin(abs(HFSS_FREQS - mhz)))
    sc = ax.scatter((AZ + 180) % 360 - 180, EL, c=F_TERRAIN[:, k], s=12, cmap='cividis')
    fig.colorbar(sc, ax=ax, label='terrain fraction')
    ax.set_xlabel('az [deg]'); ax.set_ylabel('el [deg]'); ax.set_title('%.1f MHz' % HFSS_FREQS[k])
plt.show()

## 4. Data against the model

Correlator counts include receiver gain and noise, and the transmitter's broadband noise during the raster, so the comparison is of time variation only. No gain or offset is fitted. Tooth channels (index % 8 in {7, 0, 1}) are excluded because the transmitter comb dominates them. Samples carrying an RFI bit are masked. Rows from 20:54 onward have no flags (§1). The correlations are computed on flagged rows only, and again on all rows.

In [ ]:
fig, axes = plt.subplots(nrows=2, sharex=True)
axes[0].plot(TIME, (AZ + 180) % 360 - 180, '-', ms=2, label='az (commanded)')
axes[0].plot(TIME, EL, '-', ms=2, label='el (IMU)')
axes[1].plot(TIME, D[:, ch_tx])
axes[1].plot(TIME, D[:, ch])
#axes[1].plot(TIME, 2.1e3 * T_ANT[:, ch] - 0.5e5)
#axes[1].plot(TIME, 0.3e3 * T_ANT[:, ch] + 0.3e5)
#axes[1].plot(TIME, 1.3e2 * T_ANT[:, ch] + 0.3e5)
axes[1].plot(TIME, 3.3e2 * T_ANT[:, ch] + 2.3e5)

In [ ]:
TEETH = np.abs((np.arange(1024) + 4) % 8 - 4) <= 1
use_ch = band & ~TEETH
Dm = np.where(RFI | ~use_ch[None, :], np.nan, D)
fig, axes = plt.subplots(1, 2, figsize=(17, 8), sharey=True, sharex=True, constrained_layout=True)
lo, hi = np.nanpercentile(Dm[:, use_ch], [2, 98])
m0 = axes[0].pcolormesh(FREQ, tnum, np.ma.masked_invalid(Dm), shading='nearest', norm=LogNorm(lo, hi), rasterized=True)
fig.colorbar(m0, ax=axes[0], orientation='horizontal', pad=0.08, label='counts (RFI and teeth masked)')
m1 = axes[1].pcolormesh(FREQ, tnum, np.ma.masked_invalid(np.where(use_ch[None, :], T_ANT, np.nan)),
                        shading='nearest', cmap='cividis', rasterized=True)
fig.colorbar(m1, ax=axes[1], orientation='horizontal', pad=0.08, label=r'$T_\mathrm{ant}$ [K]')
for ax, title in zip(axes, ('Data (box-air)', 'HFSS beam: 1000 K sky, 300 K terrain')):
    ax.set_xlabel('Frequency [MHz]'); ax.set_title(title); ax.set_xlim(50, 246)
axes[0].yaxis_date(); axes[0].yaxis.set_major_formatter(mdates.DateFormatter('%H:%M')); axes[0].invert_yaxis()
axes[0].set_ylabel('Time [UTC, 2026-07-17]')
plt.show()

In [ ]:
# Per channel: correlation over time between counts and T_ant, on unflagged samples.
rows = []
for c in np.flatnonzero(use_ch):
    ok = np.isfinite(Dm[:, c]) & np.isfinite(T_ANT[:, c]) & HAS_FLAGS
    ok_all = np.isfinite(Dm[:, c]) & np.isfinite(T_ANT[:, c])
    if ok.sum() < 50:
        continue
    rows.append(dict(channel=c, freq_mhz=FREQ[c], n=int(ok.sum()),
                     r=np.corrcoef(Dm[ok, c], T_ANT[ok, c])[0, 1],
                     r_all_rows=np.corrcoef(Dm[ok_all, c], T_ANT[ok_all, c])[0, 1],
                     data_frac_std=np.std(Dm[ok, c]) / np.mean(Dm[ok, c]),
                     model_frac_std=np.std(T_ANT[ok, c]) / np.mean(T_ANT[ok, c])))
CORR = pd.DataFrame(rows)
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, constrained_layout=True)
axes[0].plot(CORR.freq_mhz, CORR.r, '.', ms=3, label='rows with flags')
axes[0].plot(CORR.freq_mhz, CORR.r_all_rows, '.', ms=3, alpha=0.6, label='all rows')
axes[0].legend(fontsize=9)
axes[0].axhline(0, color='0.6', lw=1); axes[0].set_ylabel('corr(counts, $T_\\mathrm{ant}$) over time')
axes[1].semilogy(CORR.freq_mhz, CORR.data_frac_std, '.', ms=3, label='data')
axes[1].semilogy(CORR.freq_mhz, CORR.model_frac_std, '.', ms=3, label='model')
axes[1].set_ylabel('fractional std over time'); axes[1].set_xlabel('Frequency [MHz]'); axes[1].legend()
axes[0].set_title('Time variation: data against the sky-plus-terrain model (off-tooth channels)')
plt.show()
display(CORR.assign(band=pd.cut(CORR.freq_mhz, [50, 87, 108, 140, 190, 246]))
        .groupby('band', observed=True)[['r', 'r_all_rows', 'data_frac_std', 'model_frac_std']].median().round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
for ax, mhz in zip(axes, (75, 150, 200)):
    c = int(np.flatnonzero(use_ch)[np.argmin(abs(FREQ[use_ch] - mhz))])
    ok = np.isfinite(Dm[:, c])
    sc = ax.scatter(T_ANT[ok, c], Dm[ok, c], c=EL[ok], s=8, cmap='twilight')
    ax.set_xlabel(r'$T_\mathrm{ant}$ [K]'); ax.set_ylabel('counts'); ax.set_title('%.2f MHz' % FREQ[c])
fig.colorbar(sc, ax=axes, label='el [deg]')
plt.show()

## Findings

**1. Terrain takes a large, pointing-dependent share of the beam.** The beam-weighted terrain fraction runs from 0.46 to 0.76 over the raster (median 0.66). With a 1000 K sky and 300 K terrain, that gives $T_\mathrm{ant}$ of 470–678 K. It oscillates with every elevation sweep, most strongly at 100–200 MHz (about ±10%). Pointing at zenith, 56% of the 150 MHz beam lands on terrain, as expected for a front/back-symmetric beam with a horizon rising to 40°.

**2. The data follow the model where the uniform-sky assumption is reasonable.** Median correlation over time between counts and $T_\mathrm{ant}$, off-tooth channels, rows with flags:

| band | r | data std | model std |
|---|---|---|---|
| 50–87 MHz | +0.63 | 7.9% | 5.2% |
| 108–140 MHz | +0.55 | 6.8% | 8.0% |
| 140–190 MHz | −0.13 | 1.3% | 7.8% |
| 190–246 MHz | −0.07 | 2.2% | 3.2% |

At 75 MHz the counts rise with $T_\mathrm{ant}$ at about the modelled fractional rate, with distinct branches by elevation. This supports the pointing convention and the v0002 terrain, and it rules out a gross mirror or bearing error at low frequency.

**3. Above about 140 MHz, a 1000 K sky is the wrong assumption.** The sky falls roughly as $\nu^{-2.6}$. It is about 1000 K near 95 MHz, equals the 300 K terrain near 150 MHz, and is colder above that. So the real sky-to-terrain contrast shrinks and then reverses across 140–190 MHz. The correlation changes sign in exactly that band, and at 200 MHz the counts hardly vary with pointing. The transmitter's broadband noise, received through the beam, may add pointing-dependent structure near 150 MHz. That is a hypothesis and has not been tested.

**4. Terrain product.** `curation/horizon_profiles_v0002` is a full ray trace from the v0001 antenna position. It differs from v1 by a median of +0.03° (p95 0.13°, max 0.49° at 91 m). Moving the antenna around its 1.7 m bound shifts the horizon by a median of 0.27°, and up to 5.6° at narrow ridges.

**Caveats.**
- The v3-beta.1 RFI flags do not exist for 20:54–21:28 (504 of 856 rows), because the flagger's solve failed there. Where flags exist, 51% of samples carry an RFI bit, mostly `positive_auto_excess`, which on this raster is largely unmodelled ripple rather than RFI.
- Broadband dropouts (the gap-ratio mask used by the beam fits) are not masked here, and they show as low outliers.
- The HFSS beam has no ground plane or platform. The DEM is quantised to 1 m. The highline direction carries ±0.6°.

## Decision requested

1. **A frequency-dependent sky.** Replace the uniform 1000 K with a power law (for example 180 K × (ν/180 MHz)^−2.6), or with a sky map? That is a small change and would test finding 3 directly.
2. **The `fit_beam` convention (Aaron: incorrect).** Refitting the beam products (next would be v0010) in the convention used here is its own milestone. It changes the geometry parameterisation of every beam fit.
3. **Two infrastructure problems for their owners.**
   - `terrain/marjum_bundle.working_grid` shifts `marjum_dem_sw.npz` by (−1000, −1000) m twice under the current `eigsep_terrain`, so terrain sampled through it, including `marjum_lidar_constraint.py` and `lidar_explorer`, is 1 km off.
   - The v3-beta.1 RFI run is missing 20:54–21:28 on 07-17.

STOPPED AT REVIEW GATE — awaiting Aaron's approval.